# 2 · Data acquisition

| | |
|---|---|
| Lifecycle stage | 2 of 8: *Data acquisition* (collect data from APIs and the web, collect metadata) |
| Tools | Kaggle API client, `requests`, BeautifulSoup, Apache NiFi, MinIO (S3) |
| Code | `src/fraudlens/acquire.py`, `platform/nifi/build_flow.py`, `src/fraudlens/lake.py` |
| Outputs | `data/raw/*.csv` (bronze), `reports/acquisition_manifest.json`, `reports/acquisition_metadata.json`, raw layer in MinIO |

Acquisition has three jobs:

1. **Get the data** reproducibly: the official Kaggle API (or a local copy of the same files).
2. **Collect metadata**: provenance, licence, checksums, sizes, row counts and a column profile, so any
   later stage can prove it is using exactly these files.
3. **Land it in the lake** untouched (bronze / raw layer): directly with boto3, or through the
   **Apache NiFi** ingestion flow, which also writes an ingest record per file.

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path(os.environ.get("FRAUDLENS_ROOT") or (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()
CODE = Path(os.environ.get("FRAUDLENS_CODE", ROOT)).resolve()
sys.path.insert(0, str(CODE / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from fraudlens import viz
from fraudlens.utils import load_config, read_json, read_table

viz.setup()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
cfg = load_config(root=ROOT)
P = cfg["paths"]
print(f"Project root: {ROOT.name} | data present: {(P['silver'] / 'transactions.parquet').exists()}")
from fraudlens import acquire

Project root: fraudlens | data present: True


## 2.1 Source: Kaggle API

`acquire.download_from_kaggle` uses the official `kaggle` client (credentials in `~/.kaggle/kaggle.json`
or `KAGGLE_USERNAME`/`KAGGLE_KEY`). The run recorded below copied an already-downloaded local copy
(`--source local`), which produces identical files; the checksums prove it.

In [2]:
import inspect
print(inspect.getsource(acquire.download_from_kaggle))
man = read_json(P["reports"] / "acquisition_manifest.json")
{k: man[k] for k in ("dataset", "source_url", "licence", "acquired_via", "acquired_at_utc")}

def download_from_kaggle(slug: str, dest: Path) -> None:
    """Download and unzip a Kaggle dataset with the official `kaggle` API client.

    Credentials: ~/.kaggle/kaggle.json, or KAGGLE_USERNAME and KAGGLE_KEY.
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError as exc:  # pragma: no cover - depends on optional package
        raise RuntimeError("Install the Kaggle client (pip install kaggle) or use --local-dir") from exc
    api = KaggleApi()
    api.authenticate()
    log.info("Downloading %s from the Kaggle API into %s", slug, dest)
    api.dataset_download_files(slug, path=str(dest), unzip=False, quiet=False)
    for z in dest.glob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        z.unlink()



{'dataset': 'kartik2112/fraud-detection',
 'source_url': 'https://www.kaggle.com/datasets/kartik2112/fraud-detection',
 'licence': 'CC0-1.0',
 'acquired_via': 'local',
 'acquired_at_utc': '2026-10-17T15:22:31.426286+00:00'}

## 2.2 Metadata from the web: `requests` + BeautifulSoup

The dataset was produced with the open-source **Sparkov** generator. We collect its provenance (title,
description, licence) from the generator's public repository page, the way a data engineer documents
an external source. If the machine is offline the cell says so instead of failing.

In [3]:
import requests
from bs4 import BeautifulSoup

def page_metadata(url: str) -> dict:
    r = requests.get(url, timeout=15, headers={"User-Agent": "FraudLens-metadata-collector/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    meta = lambda name: (soup.find("meta", attrs={"property": name}) or soup.find("meta", attrs={"name": name}) or {}).get("content")
    return {"url": url, "status": r.status_code, "title": soup.title.get_text(strip=True) if soup.title else None,
            "description": meta("og:description") or meta("description"),
            "licence_mentioned": next((w for w in ("CC0", "MIT", "Apache", "GPL") if w in r.text), None)}

sources = ["https://github.com/namebrandon/Sparkov_Data_Generation", man["source_url"]]
rows = []
for url in sources:
    try:
        rows.append(page_metadata(url))
    except Exception as exc:
        rows.append({"url": url, "status": f"unreachable ({type(exc).__name__})"})
pd.DataFrame(rows)

,url,status,title,description,licence_mentioned
0,https://github.com/namebrandon/Sparkov_Data_Ge...,200,GitHub - namebrandon/Sparkov_Data_Generation: ...,Synthetic Credit Card Transaction Generator us...,MIT
1,https://www.kaggle.com/datasets/kartik2112/fra...,200,Credit Card Transactions Fraud Detection Datas...,Simulated Credit Card Transactions generated u...,CC0


## 2.3 Integrity: checksums, sizes and row counts

The manifest stores a SHA-256 checksum per file. Recomputing it here proves the bronze files have not
changed since acquisition (the verification stage does the same on every run).

In [4]:
rows = []
for split, f in man["files"].items():
    path = P["raw"] / f["file"]
    if path.exists():
        sha = acquire.sha256sum(path)
        rows.append({"split": split, "file": f["file"], "rows": f["rows"], "MB": round(f["bytes"] / 1e6, 1),
                     "sha256 (first 16)": sha[:16], "matches manifest": sha == f["sha256"]})
    else:
        rows.append({"split": split, "file": f["file"], "rows": f["rows"], "matches manifest": "file not present"})
pd.DataFrame(rows)

,split,file,rows,MB,sha256 (first 16),matches manifest
0,train,fraudTrain.csv,1296675,351.2000,fd7139200dbfcbed,True
1,test,fraudTest.csv,555719,150.4000,12d553ab19440c75,True


## 2.4 Column profile (metadata collection)

A profile of the raw columns from a 100,000-row sample: inferred type, missing values, distinct values.
It already tells us what preparation must deal with: a leftover index column (`Unnamed: 0`), timestamps
stored as text, card numbers stored as integers, and direct identifiers (`first`, `last`, `street`) that
must not leave the raw layer.

In [5]:
meta = read_json(P["reports"] / "acquisition_metadata.json")
print(meta["description"])
prof = pd.DataFrame(meta["columns"])
PII = {"cc_num", "first", "last", "street", "dob", "zip", "lat", "long", "job"}
prof["governance"] = prof["column"].map(lambda c: "PII" if c in PII else "")
prof.loc[prof["governance"] == "PII", "example"] = "(hidden)"
prof

Simulated credit card transactions (Sparkov generator), 1 Jan 2019 - 31 Dec 2020, 1,000 customers and 800 merchants, labelled is_fraud.


,column,inferred_dtype,null_pct_in_sample,distinct_in_sample,example,governance
0,Unnamed: 0,int64,0.0000,100000,0,
1,trans_date_trans_time,str,0.0000,98825,2019-01-01 00:00:18,
2,cc_num,int64,0.0000,922,(hidden),PII
3,merchant,str,0.0000,693,"fraud_Rippin, Kub and Mann",
4,category,str,0.0000,14,misc_net,
5,amt,float64,0.0000,21184,4.97,
6,first,str,0.0000,341,(hidden),PII
7,last,str,0.0000,470,(hidden),PII
8,gender,str,0.0000,2,F,
9,street,str,0.0000,922,(hidden),PII


In [6]:
raw_path = P["raw"] / man["files"]["train"]["file"]
safe_cols = ["trans_date_trans_time", "merchant", "category", "amt", "gender", "city", "state", "city_pop",
             "unix_time", "merch_lat", "merch_long", "is_fraud"]
if raw_path.exists():
    display(pd.read_csv(raw_path, nrows=5, usecols=safe_cols))

,trans_date_trans_time,merchant,category,amt,gender,city,state,city_pop,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,"fraud_Rippin, Kub and Mann",misc_net,4.9700,F,Moravian Falls,NC,3495,1325376018,36.0113,-82.0483,0
1,2019-01-01 00:00:44,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.2300,F,Orient,WA,149,1325376044,49.1590,-118.1865,0
2,2019-01-01 00:00:51,fraud_Lind-Buckridge,entertainment,220.1100,M,Malad City,ID,4154,1325376051,43.1507,-112.1545,0
3,2019-01-01 00:01:16,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.0000,M,Boulder,MT,1939,1325376076,47.0343,-112.5611,0
4,2019-01-01 00:03:06,fraud_Keeling-Crist,misc_pos,41.9600,M,Doe Hill,VA,99,1325376186,38.6750,-78.6325,0


## 2.5 Landing in the lake: MinIO and Apache NiFi

On the Docker platform the raw files land in the **raw (bronze) bucket** of MinIO, one folder per file so
the crawler can register each as an external table:

```
s3://fraudlens-raw/sparkov/train/fraudTrain.csv
s3://fraudlens-raw/sparkov/test/fraudTest.csv
s3://fraudlens-raw/sparkov/_manifest/manifest.json
s3://fraudlens-raw/sparkov/_ingest_log/<file>.json      <- NiFi ingest records
```

With `platform.ingest: nifi` (or `--ingest nifi`), the acquisition stage drops the files into
`data/landing/`, and the NiFi flow built by `platform/nifi/build_flow.py` does the rest:

```
ListFile(/landing) -> FetchFile -> UpdateAttribute(lake.key, ingested.at)
    -> PutS3Object(s3://fraudlens-raw/<lake.key>)
    -> AttributesToJSON -> PutS3Object(s3://fraudlens-raw/sparkov/_ingest_log/<file>.json)
```

NiFi writes with a **write-only service account** (`fraudlens-ingest`, see `platform/minio/policies/ingest.json`)
that can put objects under `sparkov/` and nothing else; the raw bucket is **versioned**, so a source file can
never be silently replaced. The acquisition stage waits until every file has arrived with the right size and
copies NiFi's ingest records into the manifest.

In [7]:
from fraudlens import lake
if lake.available(cfg):
    keys = lake.list_keys(cfg, cfg["platform"]["buckets"]["raw"], "sparkov/")
    display(pd.DataFrame({"object": keys}))
    if man.get("lake", {}).get("nifi_ingest_records"):
        display(pd.DataFrame(man["lake"]["nifi_ingest_records"]).T)
else:
    print("The Docker platform is not running here, so the lake is not reachable; the pipeline ran in local mode.")
    pub = P["reports"] / "lake_publication.json"
    print("Lake publication status:", read_json(pub).get("status") if pub.exists() else "not run")
    print("The platform workflow (deploy/github/workflows/platform.yml) runs this stage through NiFi and MinIO.")

10:40:28 | INFO    | Data lake not reachable at http://localhost:9000 (EndpointConnectionError); continuing locally


The Docker platform is not running here, so the lake is not reachable; the pipeline ran in local mode.
Lake publication status: skipped
The platform workflow (deploy/github/workflows/platform.yml) runs this stage through NiFi and MinIO.


## 2.6 Summary

* Two CSV files, 1,852,394 transactions, 501 MB, licence CC0, checksums recorded and re-verified.
* Provenance and a column profile are stored as JSON next to the data (`reports/`).
* The raw layer is immutable (versioned bucket, write-only ingest account), and NiFi records an ingest
  record per file for lineage.

**Next:** [03 · Data preparation](03_data_preparation.ipynb)